In [ ]:
from openai import OpenAI
openai_client = OpenAI()

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [ ]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

In [ ]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

courses_raw

In [ ]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

In [ ]:
"""
We start with R in RAG, which is Retrieval, but he calls it SEARCH.
"""

from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [ ]:
question = 'I just discovered the course. Can I join now?' # copying from above just so we have it clearly all in one place
index.search(question)  # returns questions from all of the courses, llm_zoomcamp, AI Dev Zoomcamp, Data Engineering Zoomcamp, etc.

In [ ]:
# We have now covered the search function, so let's add it
def search(question, course='llm-zoomcamp'):
    boost_dict={'question':2.0, 'section':0.5}
    filter_dict={'course':course}
    
    return index.search(question,
                        boost_dict=boost_dict,
                        filter_dict=filter_dict,
                        num_results=5)

In [ ]:
"""
Moving on to A in RAG, which is Augmentation, but he calls it BUILDING PROMPT.
"""

INSTRUCTIONS = f'''Your task is to answer questions from the course participants based on the provided context. 

Use the context to find relevant information and provide acurate answers. 

If the answer is not found in the context, respond with "I do not know"
'''

In [ ]:
USER_PROMPT_TEMPLATE = '''
Question: 
{question}

Context:
{context}
'''

In [ ]:
# Next up, we turn the python dictionaries that we got from our search into one long string, this makes it
# easier for the LLM to read and understand.
# I'll copy over a function that I wrote before, it takes the search_results which is the two json dicitonaries 
# and it turns them into one long string. 

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["course"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip() 

build_context(search_results)



In [ ]:
# Then we feed this long string into our prompt template that we saw before, that we will send to the llm. 
# Say search_result we got from part one, build_context that turns it into a long string 
# and the prompt, which simply formats the long string using the USER_PROMPT_TEMPLATE. 
# And we remove any trailing or leading whitespace with strip. 
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()


prompt = build_prompt(question, search_results)

print(prompt)

In [ ]:
"""
Then the last part of RAG is Generation, which is the llm. It takes the prompt we built and generates an answer, 
hence generation.
"""

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [ ]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

In [ ]:
# We can now put this together into an updated llm function.
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text


In [ ]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer


# answer = rag("I just discovered the course. Can I join now?")
answer = rag("How do I connect to Olama?")
print(answer)

In [ ]:
# ----- AGentic rag starts

In [ ]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}